# 02. MNIST Digit Clustering with Unsupervised K-Means

## 📌 Bài Toán Phân Nhóm Giả Định (Unsupervised Handwritten Digit Clustering)

> **Giả định**: Giả sử chúng ta thu thập được hàng ngàn bức ảnh chữ số viết tay nhưng **hoàn toàn KHÔNG có nhãn (unlabeled data)**. Chúng ta không biết bức ảnh nào là số 0, số 1 hay số 9.
>
> **Mục tiêu**: Sử dụng thuật toán **K-Means Clustering** vừa tự cài đặt để tự động gom các bức ảnh có nét viết tương đồng nhau vào cùng $K = 10$ nhóm (tương ứng với 10 chữ số từ $0 \to 9$).

---

## 📊 Quy Trình Xử Lý Dữ Liệu Ảnh Trong Machine Learning

1. **Ma trận ảnh 2D $\to$ Vectơ 1D (Flattening)**:
   - Mỗi bức ảnh kích thước $8 \times 8$ pixels được phẳng hóa thành 1 vectơ đặc trưng $d = 64$.
   - Giá trị mỗi pixel biểu diễn độ sáng trong khoảng $[0, 16]$.
2. **Chuẩn hóa dữ liệu (Feature Scaling)**:
   - Chia cho 16.0 để các pixel nằm trong khoảng $[0, 1]$.
3. **Gom cụm bằng K-Means**:
   - Huấn luyện `KMeans(n_clusters=10)` trên ma trận ảnh.
4. **Trực quan hóa Tâm Cụm (Centroid Image Visualization)**:
   - Biến đổi ngược các tâm cụm (1D vector $d=64$) thành dạng ảnh 2D ($8 \times 8$) để xem "hình dáng đại diện" mà K-Means tự học được!

## 🛠️ 1. Khởi Tạo Môi Trường & Load Bộ Dữ Liệu MNIST Digits

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.metrics import adjusted_rand_score, silhouette_score

# Thiết lập đường dẫn import src
current_dir = Path.cwd()
ml_learning_dir = (
    current_dir.parent.parent
    if current_dir.name in ["supervised", "unsupervised"]
    else current_dir.parent
)
src_dir = ml_learning_dir / "src"
if str(src_dir) not in sys.path:
    sys.path.append(str(src_dir))

from ai_journey_shared.plotting import save_figure, setup_plot_style  # noqa: E402
from ai_journey_shared.utils import set_seed  # noqa: E402
from ml_learning.unsupervised.kmeans import KMeans  # noqa: E402

setup_plot_style()
set_seed(42)

# Load bộ dữ liệu MNIST Digits dạng tuple (X, y) chuẩn kiểu cho BasedPyright
X_raw_data, y_true_data = load_digits(return_X_y=True)
X_raw = np.asarray(X_raw_data, dtype=np.float64)
y_true = np.asarray(y_true_data, dtype=np.int64)

print(f"✅ Đã load thành công {X_raw.shape[0]} bức ảnh MNIST Digits!")
print(f"📐 Kích thước ma trận đặc trưng X: {X_raw.shape} (1797 mẫu, 64 pixels/ảnh)")

## 🖼️ 2. Hiển Thị Mẫu Ảnh Viết Tay Ban Đầu

In [ ]:
# Hiển thị 10 bức ảnh mẫu ngẫu nhiên từ tập dữ liệu
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    # Reshape từ 1D (64) thành 2D (8x8) để hiển thị ảnh
    img_2d = X_raw[i].reshape(8, 8)
    ax.imshow(img_2d, cmap="binary")
    ax.set_title(f"Mẫu #{i}")
    ax.axis("off")

plt.suptitle("Một Số Ảnh Chữ Số Viết Tay Ban Đầu (Chưa Có Nhãn)", fontsize=14)
plt.tight_layout()
plt.show()

## ⚙️ 3. Chuẩn Hóa Dữ Liệu (Normalization)

K-Means tính khoảng cách Euclidean giữa các pixel. Ta chuẩn hóa giá trị pixel về khoảng $[0, 1]$ bằng cách chia cho 16.0.

In [ ]:
X_scaled = X_raw / 16.0
print(f"Pixel min: {X_scaled.min()}, Pixel max: {X_scaled.max()}")

## 🚀 4. Huấn Luyện K-Means Vừa Tự Viết (`n_clusters = 10`)

Bây giờ ta sẽ cho thuật toán `KMeans` phân chia 1,797 bức ảnh thành 10 nhóm hoàn toàn tự động.

In [ ]:
try:
    # Khởi tạo K-Means với K = 10 cụm
    kmeans = KMeans(n_clusters=10, max_iter=300, tol=1e-4)
    kmeans.fit(X_scaled)
    cluster_labels = kmeans.predict(X_scaled)
    centroids = kmeans.centroids

    if centroids is not None:
        print("🎉 Huấn luyện K-Means phân nhóm MNIST thành công!")
        print(f"Centroids shape: {centroids.shape} (10 tâm, 64 pixels)")

except Exception as e:
    print(f"❌ Lỗi: {e}")

## 🔍 5. TRỰC QUAN HÓA TÂM CỤM (CENTROID IMAGES)

**Điều Kỳ Diệu Của Unsupervised Learning**:
Bằng cách đưa các tâm cụm (kích thước 64) về lại dạng ảnh $8 \times 8$, chúng ta có thể nhìn thấy bức ảnh "trung bình đại diện" mà K-Means tự khám phá ra cho từng nhóm!

In [ ]:
if kmeans.centroids is not None:
    fig, axes = plt.subplots(2, 5, figsize=(10, 5))
    for k, ax in enumerate(axes.flat):
        # Reshape tâm cụm k từ 64 về ảnh 8x8
        centroid_img = kmeans.centroids[k].reshape(8, 8)
        ax.imshow(centroid_img, cmap="binary")
        ax.set_title(f"Cụm #{k}")
        ax.axis("off")

    plt.suptitle("Hình Dạng Chữ Số Đại Diện Do K-Means Tự Học Được (10 Tâm Cụm)", fontsize=14)
    fig_path = ml_learning_dir / "figures" / "mnist_kmeans_centroids.png"
    save_figure(fig, fig_path)
    print(f"🖼️ Đã lưu ảnh tâm cụm vào {fig_path}")

## 📈 6. Đánh Giá Chất Lượng Phân Nhóm

1. **Silhouette Score**: Đánh giá độ co cụm và tách biệt giữa các nhóm (không cần nhãn thực tế).
2. **Adjusted Rand Index (ARI)**: So sánh cụm K-Means tìm được với nhãn thực tế $y_{true}$ (càng gần 1.0 càng giống phân loại của con người).

In [ ]:
sil_score = silhouette_score(X_scaled, cluster_labels)
ari_score = adjusted_rand_score(y_true, cluster_labels)

print(f"📊 Silhouette Score: {sil_score:.4f} (Đo độ rõ ràng của các cụm)")
print(f"📊 Adjusted Rand Index (ARI): {ari_score:.4f} (So sánh với nhãn thực tế 0-9)")